In [1]:
from langchain_core.documents import Document
documents = [
    Document(page_content="Dogs are great companions, know for their loyalty and playfulness.", metadata={"source": "mammal-pets-doc"}),
    Document(page_content="Cats are independent and curious animals, often kept as pets.", metadata={"source": "mammal-pets-doc"}),
    Document(page_content="Birds are colorful and can mimic human speech.", metadata={"source": "mammal-pets-doc"}),
    Document(page_content="Fish are aquatic animals that come in various shapes and sizes.", metadata={"source": "mammal-pets-doc"}),
    Document(page_content="Reptiles are cold-blooded animals, including snakes and lizards.", metadata={"source": "mammal-pets-doc"})
]

In [15]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm= ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=groq_api_key)

In [16]:
## embeddings using hugging face
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6755.80it/s]


In [17]:
## Vectorstores
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(documents, embedding=embeddings)
vectorstore


In [18]:
vectorstore.similarity_search("cats")

[Document(id='421f2cfa-e4f3-41ba-b6f7-5653321f385c', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent and curious animals, often kept as pets.'),
 Document(id='ad76f7e4-667b-4937-b0bf-0fbb7c46742a', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent and curious animals, often kept as pets.'),
 Document(id='89ca0e14-0261-42f0-a8ad-7024380383f1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent and curious animals, often kept as pets.'),
 Document(id='9fbb3d5e-6ca2-44cf-a2b9-44df94b1187b', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, know for their loyalty and playfulness.')]

In [19]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cats", "dogs", "birds"])


[[Document(id='89ca0e14-0261-42f0-a8ad-7024380383f1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent and curious animals, often kept as pets.')],
 [Document(id='7ac15bd1-b612-4358-9085-cbf0b7bf12bf', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, know for their loyalty and playfulness.')],
 [Document(id='789d84a6-d4ac-40ba-9dc7-ba71f67eeb32', metadata={'source': 'mammal-pets-doc'}, page_content='Birds are colorful and can mimic human speech.')]]

In [21]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})
retriever.batch(["cats", "dogs", "birds"])

[[Document(id='89ca0e14-0261-42f0-a8ad-7024380383f1', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent and curious animals, often kept as pets.')],
 [Document(id='7ac15bd1-b612-4358-9085-cbf0b7bf12bf', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, know for their loyalty and playfulness.')],
 [Document(id='789d84a6-d4ac-40ba-9dc7-ba71f67eeb32', metadata={'source': 'mammal-pets-doc'}, page_content='Birds are colorful and can mimic human speech.')]]

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message ="""
Answer the questions using the provided context only.
{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain={"context": retriever, "question": RunnablePassthrough()}|prompt|llm

response = rag_chain.invoke("What are cats known for?")
print(response.content)

Cats are known for being independent and curious animals.
